# The Compliance-Aware Agent

## Introduction

A model risk officer at a bank asks a simple question about your agent: *"It approved this loan. Show me why, who signed off, and how sure it was."*

Most agents cannot answer. They emit a decision and a paragraph of prose. There is no record of what the agent saw, no confidence attached to the claim, and no point in the run where a human was required to look. In a regulated environment that is not a missing feature — it is a blocker.

Regulators are explicit about this:

| Regime | What it demands of an automated decision |
| --- | --- |
| **FCA Consumer Duty** (UK) | Firms must evidence that outcomes are fair — which means reconstructing *why* a decision was reached |
| **MiFID II** Art. 16(7) (EU) | Records of decisions must be reconstructable and retained; algorithmic trading needs pre-trade controls |
| **Basel III** BCBS 239 | Risk data must be accurate, complete, and traceable to its source |
| **SR 11-7** (US Fed) | Models need documented validation, and *"effective challenge"* — a human able to overturn the model |

Three properties follow from that table, and a standard agent has none of them:

1. **No audit trail.** The reasoning lives in a chat transcript that is thrown away. There is nothing to hand an examiner.
2. **No escalation logic.** The agent acts with the same authority on "reformat this date field" and "wire $12M". Nothing forces a human into the loop.
3. **Binary pass/fail with no confidence signalling.** The agent says "compliant". It does not say *"compliant, but I am 71% sure and two of my checks disagreed"* — which is exactly the case a human needs to see.

This notebook builds an agent that has all three. We will:

1. Make risk a **first-class primitive** — an FC-A/B/C/D failure budget that every decision is scored against, rather than a comment in a prompt.
2. Wire up a **three-layer architecture** — an intent router, three specialist sub-agents, and a reconciliation agent that merges their findings and flags conflicts.
3. Enforce a **human-in-the-loop checkpoint** that physically blocks the agent's tool call until a human approves it, using the SDK's `can_use_tool` permission callback.
4. Write every step to an **append-only audit trail** that a database trigger prevents anyone from rewriting.

Then we run a fictional loan application and a trade instruction end-to-end, and watch one escalate to a human and the other get blocked outright.

> **Where this sits in the series:** notebook [03](https://github.com/anthropics/claude-cookbooks/blob/main/claude_agent_sdk/03_The_site_reliability_agent.ipynb) gave an agent write access to infrastructure and used hooks to keep it safe. This notebook asks the harder governance question: not *"can the agent act safely?"* but *"can you prove to a regulator how it decided, and stop it when confidence runs out?"*

## Step 0: Environment Setup

Create a `.env` file in this directory with your Anthropic API key:

```
ANTHROPIC_API_KEY=your-key-here
```

In [1]:
%%capture
%pip install claude-agent-sdk python-dotenv

In [2]:
import asyncio
import json
import sqlite3
import textwrap
from collections.abc import AsyncIterator
from dataclasses import dataclass, field
from datetime import UTC, datetime
from enum import IntEnum
from typing import Any

from dotenv import load_dotenv

from claude_agent_sdk import (
    ClaudeAgentOptions,
    PermissionResultAllow,
    PermissionResultDeny,
    ResultMessage,
    ThinkingConfigDisabled,
    ToolPermissionContext,
    create_sdk_mcp_server,
    query,
    tool,
)

load_dotenv()

# The orchestrator does the judgement-heavy work: routing and reconciling conflicts.
ORCHESTRATOR_MODEL = "claude-sonnet-5"
# Specialists are narrow and high-volume — Haiku keeps them cheap.
SPECIALIST_MODEL = "claude-haiku-4-5"

The SDK resolves credentials for you: it uses `ANTHROPIC_API_KEY` when set, and otherwise falls back to an existing Claude Code CLI login. The check below is a warning rather than a hard failure so the notebook still runs under either.

In [3]:
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    print(
        "ANTHROPIC_API_KEY is not set — the Agent SDK will fall back to your Claude Code CLI login.\n"
        "If neither is available, add the key to a .env file in this directory."
    )
else:
    print("ANTHROPIC_API_KEY found.")

ANTHROPIC_API_KEY is not set — the Agent SDK will fall back to your Claude Code CLI login.
If neither is available, add the key to a .env file in this directory.


## Step 1: The Failure Budget Framework

Every agent has a failure budget, whether or not anyone writes it down. The question is only how much damage an incorrect action can do before someone catches it. Most agents leave this implicit in a system prompt (*"be careful with financial data"*), which means it is unenforceable and invisible in review.

We make it a **first-class primitive** instead: every decision carries an explicit tier that determines what the agent is allowed to do next.

| Tier | Meaning | Condition | Agent may act? |
| --- | --- | --- | --- |
| **FC-A** | Fully automated | confidence > 0.95 **and** low-stakes | Yes, silently |
| **FC-B** | Automated with logging | confidence > 0.85 | Yes, but recorded for later review |
| **FC-C** | Human review required | confidence 0.70–0.85 **or** medium-stakes | Only after a human approves |
| **FC-D** | Block and escalate | confidence < 0.70 **or** high-stakes/regulated | No — halt immediately |

Two independent signals feed the tier, and keeping them separate is the whole design:

- **Confidence** is what the model reports about *its own* certainty. It is a model output.
- **Stakes** are a property of *the action*, and — this is the important part — **the model does not get a vote.** Executing a £12M trade is REGULATED-stakes whether the agent is 55% or 99% sure. SR 11-7's "effective challenge" principle exists precisely because a confident model can be a confidently wrong one, so the authority to act cannot route through the model's own confidence.

Do not confuse stakes with *risk*. A risky loan application and a routine one carry the **same** stakes — a credit-underwriting decision — because stakes describe what the *action* can do, not how good the input looks. Risk is something the specialists assess; stakes are fixed by what class of action this is. A high-risk application still only needs a human to look (FC-C); it is not auto-blocked (FC-D). Auto-blocking is reserved for actions that must never be automated at all, like executing a trade.

Because stakes are a property of the action class, we set them by **policy**, not by asking the model per document. A firm decides once that "a trade instruction is REGULATED" and "a loan application is a MEDIUM-stakes underwriting recommendation" — and every document of that class inherits the floor. This is both more defensible (an examiner can read the policy) and more deterministic (the tier does not swing with the model's mood).

So the tier is the stricter of two signals: the confidence-derived tier, and the policy stakes floor. Encoding the tiers as an `IntEnum` makes "stricter" a plain `max()`.

In [4]:
class FCTier(IntEnum):
    """Failure budget tiers, ordered by severity so that max() picks the stricter one."""

    A = 0  # fully automated
    B = 1  # automated, logged
    C = 2  # human review required before acting
    D = 3  # blocked, escalate immediately

    @property
    def label(self) -> str:
        return f"FC-{self.name}"

    @property
    def requires_human(self) -> bool:
        return self >= FCTier.C


class Stakes(IntEnum):
    """How much damage the action itself could do, independent of model confidence."""

    LOW = 0
    MEDIUM = 1
    HIGH = 2
    REGULATED = 3


# Stakes set a floor the model's confidence can never argue its way below.
_STAKES_FLOOR = {
    Stakes.LOW: FCTier.A,
    Stakes.MEDIUM: FCTier.C,
    Stakes.HIGH: FCTier.D,
    Stakes.REGULATED: FCTier.D,
}


def _confidence_tier(confidence: float) -> FCTier:
    if confidence > 0.95:
        return FCTier.A
    if confidence > 0.85:
        return FCTier.B
    if confidence >= 0.70:
        return FCTier.C
    return FCTier.D


def assign_tier(confidence: float, stakes: Stakes) -> FCTier:
    """Assign a failure budget tier from model confidence and the stakes of the action.

    The two signals are computed independently and the stricter one wins.
    """
    if not 0.0 <= confidence <= 1.0:
        raise ValueError(f"confidence must be in [0, 1], got {confidence}")
    return max(_confidence_tier(confidence), _STAKES_FLOOR[stakes])


# Firm policy: the stakes floor for each class of document. Set once, by humans,
# and applied to every document of that class — not judged ad hoc by the model.
POLICY_FLOOR: dict[str, Stakes] = {
    "trade_instruction": Stakes.REGULATED,  # executes a trade / moves client money
    "loan_application": Stakes.MEDIUM,  # underwriting recommendation a human ratifies
    "kyc_review": Stakes.MEDIUM,  # changes a client record
    "other": Stakes.LOW,  # read-only / informational
}

The table below is the whole framework in six rows. Read rows 3 and 4 together: identical 0.99 confidence, opposite outcomes, because the stakes floor overrides a confident model. And notice that a *low-confidence read* of a low-stakes document (row 6) escalates on the confidence signal alone — you do not need high stakes to pull a human in.

In [5]:
_examples = [
    (0.98, Stakes.LOW, "Reformat a date field in an internal report"),
    (0.90, Stakes.LOW, "Summarise a customer's account history"),
    (0.99, Stakes.MEDIUM, "Record a loan underwriting recommendation"),
    (0.99, Stakes.REGULATED, "Submit a trade instruction to the venue"),
    (0.80, Stakes.MEDIUM, "Recommend a decision on a borderline mortgage"),
    (0.55, Stakes.LOW, "Interpret an ambiguous source-of-funds note"),
]

print(f"{'confidence':>10}  {'stakes':<10}  {'tier':<6}  {'human?':<7}  action")
print("-" * 96)
for _confidence, _stakes, _action in _examples:
    _tier = assign_tier(_confidence, _stakes)
    _human = "yes" if _tier.requires_human else "no"
    print(f"{_confidence:>10.2f}  {_stakes.name:<10}  {_tier.label:<6}  {_human:<7}  {_action}")

confidence  stakes      tier    human?   action
------------------------------------------------------------------------------------------------
      0.98  LOW         FC-A    no       Reformat a date field in an internal report
      0.90  LOW         FC-B    no       Summarise a customer's account history
      0.99  MEDIUM      FC-C    yes      Record a loan underwriting recommendation
      0.99  REGULATED   FC-D    yes      Submit a trade instruction to the venue
      0.80  MEDIUM      FC-C    yes      Recommend a decision on a borderline mortgage
      0.55  LOW         FC-D    yes      Interpret an ambiguous source-of-funds note


## Step 2: The Append-Only Audit Trail

An audit trail that can be edited is not an audit trail. If a row can be silently updated after the fact, it evidences nothing — which is the practical meaning of BCBS 239's traceability requirement and MiFID II's demand that decisions be *reconstructable*.

So we do not rely on the application politely refraining from writing. We enforce append-only **in the database**, with triggers that raise on any `UPDATE` or `DELETE`. Application code cannot opt out, and a bug cannot quietly erase a decision.

Each row captures the six things an examiner asks for: **when**, **which agent**, **what it saw**, **what it concluded**, **how sure it was**, and **what tier that put it in** — plus the human's decision where one was required.

> We use an in-memory SQLite database so the notebook is self-contained. In production this is a durable, access-controlled store — but the trigger pattern is exactly the same.

In [6]:
class AuditLog:
    """Append-only decision log. UPDATE and DELETE are blocked by database triggers."""

    def __init__(self, path: str = ":memory:") -> None:
        self.conn = sqlite3.connect(path)
        self.conn.row_factory = sqlite3.Row
        self.conn.executescript(
            """
            CREATE TABLE IF NOT EXISTS audit (
                id            INTEGER PRIMARY KEY AUTOINCREMENT,
                ts            TEXT    NOT NULL,
                case_id       TEXT    NOT NULL,
                agent         TEXT    NOT NULL,
                input         TEXT    NOT NULL,
                output        TEXT    NOT NULL,
                confidence    REAL,
                fc_tier       TEXT    NOT NULL,
                human_decision TEXT
            );

            -- The audit trail is evidence: once written, a row is immutable.
            CREATE TRIGGER IF NOT EXISTS audit_no_update
            BEFORE UPDATE ON audit
            BEGIN
                SELECT RAISE(ABORT, 'audit trail is append-only: UPDATE is forbidden');
            END;

            CREATE TRIGGER IF NOT EXISTS audit_no_delete
            BEFORE DELETE ON audit
            BEGIN
                SELECT RAISE(ABORT, 'audit trail is append-only: DELETE is forbidden');
            END;
            """
        )
        self.conn.commit()

    def record(
        self,
        *,
        case_id: str,
        agent: str,
        input_summary: str,
        output: Any,
        fc_tier: FCTier,
        confidence: float | None = None,
        human_decision: str | None = None,
    ) -> int:
        """Append one decision. Returns the new row id."""
        cursor = self.conn.execute(
            "INSERT INTO audit (ts, case_id, agent, input, output, confidence, fc_tier,"
            " human_decision) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
            (
                datetime.now(UTC).isoformat(timespec="seconds"),
                case_id,
                agent,
                input_summary,
                json.dumps(output, default=str),
                confidence,
                fc_tier.label,
                human_decision,
            ),
        )
        self.conn.commit()
        return cursor.lastrowid

    def rows(self, case_id: str | None = None) -> list[sqlite3.Row]:
        if case_id is None:
            return self.conn.execute("SELECT * FROM audit ORDER BY id").fetchall()
        return self.conn.execute(
            "SELECT * FROM audit WHERE case_id = ? ORDER BY id", (case_id,)
        ).fetchall()

    def render(self, case_id: str | None = None) -> None:
        """Print the trail the way a reviewer would read it."""
        for row in self.rows(case_id):
            human = f"  |  human: {row['human_decision']}" if row["human_decision"] else ""
            confidence = f"{row['confidence']:.2f}" if row["confidence"] is not None else "  — "
            print(
                f"[{row['id']:>2}] {row['ts']}  {row['agent']:<22} conf={confidence}  "
                f"{row['fc_tier']}{human}"
            )
            print(f"     in : {textwrap.shorten(row['input'], 88)}")
            print(f"     out: {textwrap.shorten(row['output'], 88)}")


audit = AuditLog()

Before trusting the trail, prove the guarantee actually holds. We write a row, then try to tamper with it the way a buggy retry or a bad actor would.

In [7]:
_probe_id = audit.record(
    case_id="SELFTEST",
    agent="self-test",
    input_summary="verify the append-only guarantee",
    output={"note": "this row must be immutable"},
    fc_tier=FCTier.A,
    confidence=1.0,
)

for _sql, _description in [
    ('UPDATE audit SET output = \'{"note": "tampered"}\' WHERE id = ?', "UPDATE"),
    ("DELETE FROM audit WHERE id = ?", "DELETE"),
]:
    try:
        audit.conn.execute(_sql, (_probe_id,))
    except sqlite3.IntegrityError as exc:
        print(f"{_description} blocked -> {exc}")
    else:
        raise AssertionError(f"{_description} was NOT blocked — the trail is not append-only!")

UPDATE blocked -> audit trail is append-only: UPDATE is forbidden
DELETE blocked -> audit trail is append-only: DELETE is forbidden


## Step 3: The Three-Layer Architecture

A single agent asked to "check this document for compliance" gives you one opinion with one confidence score, and no way to tell a confident answer from a lucky one. Splitting the work across independent specialists gives us something far more useful: **disagreement**.

If the document extractor is sure the applicant earns £48,000 and the risk assessor is sure the file shows £95,000, that conflict is the single most valuable signal in the entire run — and it only exists because two agents looked separately.

```
                       ┌──────────────────────────────┐
   document ─────────▶ │  Layer 1: Intent Router      │   claude-sonnet-5
                       │  classify + provisional tier │
                       └───────────────┬──────────────┘
                                       │
              ┌────────────────────────┼────────────────────────┐
              ▼                        ▼                        ▼
   ┌────────────────────┐  ┌────────────────────┐  ┌────────────────────┐
   │ document_extractor │  │  compliance_check  │  │  risk_assessor     │  claude-haiku-4-5
   │ facts + confidence │  │ rule breaches      │  │ exposure + red     │  (independent calls)
   │                    │  │                    │  │ flags              │
   └─────────┬──────────┘  └─────────┬──────────┘  └─────────┬──────────┘
             └───────────────────────┼───────────────────────┘
                                     ▼
                       ┌──────────────────────────────┐
                       │  Layer 3: Reconciliation     │   claude-sonnet-5
                       │  merge, flag conflicts,      │
                       │  set FINAL tier              │
                       └───────────────┬──────────────┘
                                       ▼
                        FC-A/B ──▶ act        FC-C ──▶ human approval
                                              FC-D ──▶ blocked
```

Every arrow in that diagram writes a row to the audit trail.

### The shared agent runner

All three layers use the same primitive: a `query()` call pinned to a model, given a JSON schema, and logged. Passing `output_format={"type": "json_schema", ...}` makes the SDK return a validated object on `ResultMessage.structured_output` — so we get a real `confidence` float to feed `assign_tier()`, instead of regex-ing a number out of prose.

Four options on that `query()` call are load-bearing, and each one is a lesson the first draft of this notebook learned the hard way:

- **`disallowed_tools=[...]`** — these agents reason over text we hand them; they have no business touching the filesystem or the web. But `allowed_tools=[]` does **not** remove the built-in tools — it only declines to *pre-approve* them. Left alone, the agent still sees `Read`, `Bash`, `Grep`, etc., and will happily try to call them, burning its turn budget on tools that go nowhere. `disallowed_tools` actually takes them off the table.
- **`thinking=ThinkingConfigDisabled(...)`** — this is the non-obvious one. The SDK implements structured output as a `StructuredOutput` *tool* the model calls to emit its object. With adaptive thinking on, the model treats that as an invitation to *keep refining* — it calls the tool, thinks, calls it again, five or six times — and runs out of turns before it ever settles, failing the whole step. Disabling thinking makes it emit once and stop. These are structured-extraction steps, not open-ended reasoning, so this costs us nothing and buys determinism.
- **`max_turns=6`** — headroom for the reason-then-emit round trip plus any stray tool attempt. `max_turns=1` cuts the model off before the schema is ever filled; too low and a chatty preamble starves it. Six is comfortable margin for a step that should take two.
- **`allowed_tools=[]`** — no tool is pre-approved. The only agent that reaches a tool is the one behind the human checkpoint in Step 5, and it goes through the gate.

**One more thing, and it is the difference between a demo and something you would run:** each agent call goes through a **timeout-and-retry** wrapper. A pipeline that makes ten model calls per document *will* occasionally see a transient one — a subprocess that exits non-zero, a call that wedges. In a compliance system a transient blip must not become a dropped decision, so `_run_structured` wraps every call in a wall-clock timeout and retries a few times with backoff. Two implementation details make the retry actually safe: we **drain the message stream fully** rather than `return`-ing from inside the `async for` (an unclosed async generator collides with the next call's cleanup — `RuntimeError: aclose(): asynchronous generator is already running`), and we treat an empty `structured_output` as a failure worth retrying, not a silent zero.

The `stakes` argument is the policy floor for the case (from `POLICY_FLOOR`), so the tier we log for each agent already reflects both signals.

In [8]:
# Built-in tools these text-only agents must never reach for. allowed_tools=[] does not
# remove them — only disallowed_tools does — and a stray tool call wastes the turn budget.
NO_BUILTIN_TOOLS = [
    "Bash",
    "Read",
    "Write",
    "Edit",
    "Glob",
    "Grep",
    "WebSearch",
    "WebFetch",
    "Task",
    "TodoWrite",
    "NotebookEdit",
]


async def _run_structured(
    model: str, system_prompt: str, prompt: str, schema: dict[str, Any]
) -> dict[str, Any] | None:
    """One schema-constrained call. Drains the stream fully so the generator closes cleanly."""
    options = ClaudeAgentOptions(
        model=model,
        system_prompt=system_prompt,
        allowed_tools=[],  # nothing is pre-approved
        disallowed_tools=NO_BUILTIN_TOOLS,  # actually take the built-in tools off the table
        thinking=ThinkingConfigDisabled(
            type="disabled"
        ),  # emit the schema object once, don't re-refine
        max_turns=6,  # room for reason-then-emit plus any stray attempt
        output_format={"type": "json_schema", "schema": schema},
    )
    # Drain to completion — do NOT return from inside the loop, or the async generator is
    # left open and its later aclose() collides with the next call's cleanup.
    result: ResultMessage | None = None
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, ResultMessage):
            result = message
    return result.structured_output if result is not None else None


async def run_agent(
    *,
    case_id: str,
    name: str,
    model: str,
    system_prompt: str,
    prompt: str,
    schema: dict[str, Any],
    input_summary: str,
    stakes: Stakes,
    attempts: int = 3,
    timeout_s: float = 150.0,
) -> dict[str, Any]:
    """Run one schema-constrained agent, with timeout + retry, and log its decision."""
    last_error = "unknown"
    output: dict[str, Any] | None = None
    for attempt in range(attempts):
        try:
            output = await asyncio.wait_for(
                _run_structured(model, system_prompt, prompt, schema), timeout=timeout_s
            )
            if output is not None:
                break
            last_error = "empty structured_output"
        except (TimeoutError, Exception) as exc:  # noqa: BLE001 — transient CLI/subprocess errors are retryable
            last_error = f"{type(exc).__name__}: {exc}"
        if attempt < attempts - 1:
            await asyncio.sleep(1.5 * (attempt + 1))  # linear backoff

    if output is None:
        raise RuntimeError(f"{name}: failed after {attempts} attempts ({last_error})")

    confidence = float(output.get("confidence", 0.0))
    tier = assign_tier(confidence, stakes)
    audit.record(
        case_id=case_id,
        agent=name,
        input_summary=input_summary,
        output=output,
        confidence=confidence,
        fc_tier=tier,
    )
    return output


def schema(properties: dict[str, Any], required: list[str]) -> dict[str, Any]:
    """Build a strict JSON schema — every agent must report a confidence."""
    return {
        "type": "object",
        "properties": {
            **properties,
            "confidence": {
                "type": "number",
                "description": "Your calibrated confidence in this output, 0.0 to 1.0.",
            },
        },
        "required": [*required, "confidence"],
        "additionalProperties": False,
    }

### Layer 1: The Intent Router

The router reads the document and decides one thing: **what class of request is this?** From the class, `POLICY_FLOOR` gives us the stakes — the router does not judge stakes itself, because (as we argued in Step 1) that is a policy decision, not a model decision. The router's own contribution to the tier is its *confidence* in the classification.

Getting the class right is high-leverage: it is what routes a trade instruction to a REGULATED floor and a loan to a MEDIUM one. So the router is a good place to spend the stronger model.

In [9]:
ROUTER_SCHEMA = schema(
    {
        "document_type": {
            "type": "string",
            "enum": list(POLICY_FLOOR),  # the classes firm policy has a stakes floor for
        },
        "intent": {"type": "string", "description": "One sentence: what is being asked for."},
        "reasoning": {"type": "string"},
    },
    ["document_type", "intent", "reasoning"],
)

ROUTER_PROMPT = """You are the intent router for a regulated financial services firm.

Classify the inbound document into exactly one type. You are not judging how risky the
document is or deciding the outcome — only identifying what kind of request it is, so the
firm's policy can attach the right level of scrutiny.

- trade_instruction: an instruction to buy, sell, or settle securities, or to move client money.
- loan_application: an application for credit — a mortgage, loan, or facility.
- kyc_review: onboarding or a change to a client's record or risk profile.
- other: anything read-only or informational.

Report calibrated confidence in your classification. If the document is ambiguous or could
plausibly be a trade or money movement, say so with a lower confidence rather than guessing."""


async def route(case_id: str, document: str) -> dict[str, Any]:
    return await run_agent(
        case_id=case_id,
        name="intent_router",
        model=ORCHESTRATOR_MODEL,
        system_prompt=ROUTER_PROMPT,
        prompt=f"Classify this document:\n\n<document>\n{document}\n</document>",
        schema=ROUTER_SCHEMA,
        input_summary=textwrap.shorten(document.strip().replace("\n", " "), 120),
        stakes=Stakes.LOW,  # the router's own log tier reflects its confidence; the policy floor is applied in the pipeline
    )

### Layer 2: The Specialist Sub-Agents

Three narrow agents, each with one job and no knowledge of the others' conclusions. That isolation is the point — it is what makes their agreement meaningful and their disagreement detectable. If they shared a context window, the second agent would anchor on the first one's answer and we would be back to a single opinion wearing three hats. They are pinned to `claude-haiku-4-5`: the work is narrow extraction and rule-checking, and this is the high-volume tier where cost actually accumulates.

We run them **sequentially** here — and it is worth being precise about why, because the obvious move is `asyncio.gather`. Each `claude-agent-sdk` `query()` is an async generator backed by its own subprocess and anyio task group; running several under `asyncio.gather` in one event loop makes their cleanup collide (`RuntimeError: aclose(): asynchronous generator is already running`) and intermittently wedges. The isolation that matters for our purpose is **context isolation** — each specialist gets its own fresh call and never sees the others' findings — and that holds whether they run in sequence or in parallel. To genuinely parallelise, run each specialist in its own worker process (or a separate agent service) and collect the results, rather than sharing one event loop.

In [10]:
SPECIALISTS = {
    "document_extractor": {
        "system_prompt": (
            "You extract structured facts from financial documents. Extract only what is "
            "literally present. Never infer, complete, or tidy up a missing value — a field "
            "that is absent must be reported as missing. Lower your confidence when the "
            "document is ambiguous, internally inconsistent, or partially illegible."
        ),
        "schema": schema(
            {
                "facts": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "field": {"type": "string"},
                            "value": {"type": "string"},
                        },
                        "required": ["field", "value"],
                        "additionalProperties": False,
                    },
                },
                "missing_fields": {"type": "array", "items": {"type": "string"}},
            },
            ["facts", "missing_fields"],
        ),
        "task": "Extract every stated fact, and list any field you would expect but cannot find.",
    },
    "compliance_check": {
        "system_prompt": (
            "You check documents against UK/EU financial regulation: FCA Consumer Duty "
            "(fair customer outcomes), MiFID II (suitability, best execution, record-keeping), "
            "and Basel III / BCBS 239 (risk data accuracy and traceability). "
            "Report only breaches you can point to evidence for in the document. "
            "If evidence is thin, say so with a low confidence rather than asserting a breach."
        ),
        "schema": schema(
            {
                "breaches": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "regulation": {"type": "string"},
                            "issue": {"type": "string"},
                            "severity": {"type": "string", "enum": ["low", "medium", "high"]},
                        },
                        "required": ["regulation", "issue", "severity"],
                        "additionalProperties": False,
                    },
                },
                "verdict": {"type": "string", "enum": ["pass", "concerns", "fail"]},
            },
            ["breaches", "verdict"],
        ),
        "task": "Identify regulatory breaches or concerns, citing the specific regime.",
    },
    "risk_assessor": {
        "system_prompt": (
            "You assess financial and conduct risk. Quantify exposure where the document "
            "supports it and flag anything anomalous: round-number transactions, urgency "
            "pressure, unexplained third parties, figures that contradict each other, or "
            "amounts inconsistent with the stated profile. Lower your confidence when the "
            "document does not give you enough to judge."
        ),
        "schema": schema(
            {
                "exposure": {
                    "type": "string",
                    "description": "Financial exposure if this is wrong.",
                },
                "red_flags": {"type": "array", "items": {"type": "string"}},
                "risk_level": {"type": "string", "enum": ["low", "medium", "high"]},
            },
            ["exposure", "red_flags", "risk_level"],
        ),
        "task": "Assess the risk of acting on this document and list any red flags.",
    },
}


async def run_specialists(case_id: str, document: str, stakes: Stakes) -> dict[str, dict[str, Any]]:
    """Run every specialist. Each gets a fresh call and never sees the others' findings."""
    findings: dict[str, dict[str, Any]] = {}
    for name, spec in SPECIALISTS.items():
        findings[name] = await run_agent(
            case_id=case_id,
            name=name,
            model=SPECIALIST_MODEL,
            system_prompt=spec["system_prompt"],
            prompt=f"{spec['task']}\n\n<document>\n{document}\n</document>",
            schema=spec["schema"],
            input_summary=spec["task"],
            stakes=stakes,
        )
    return findings

### Layer 3: The Reconciliation Agent

The reconciler is the only agent that sees all three specialist reports at once. Its job is not to re-do their work but to answer three questions a regulator would ask:

1. **Do the specialists actually agree?** Contradictions between independent agents are explicit output, not something averaged away.
2. **How confident is the *merged* picture?** This is what drives the confidence side of the tier, and it must not simply inherit a specialist's certainty — two confident specialists who contradict each other should produce a *low* merged confidence.
3. **What would a human need to know?** A one-paragraph brief for whoever picks up the escalation.

Note what the reconciler does **not** do: it does not set the stakes. Stakes come from policy (`POLICY_FLOOR`), the same floor the router's classification selected. The reconciler can push the tier up through low confidence, but it cannot talk the system down below the policy floor — `final = max(provisional, reconciled)` guarantees the tier only ratchets upward.

This runs on `claude-sonnet-5`. Reconciling contradictory evidence is the judgement-heavy step in the pipeline and the one worth paying for.

In [11]:
RECONCILER_SCHEMA = schema(
    {
        "summary": {"type": "string", "description": "What the specialists collectively found."},
        "conflicts": {
            "type": "array",
            "description": "Direct contradictions between specialists. Empty if they agree.",
            "items": {
                "type": "object",
                "properties": {
                    "between": {"type": "array", "items": {"type": "string"}},
                    "description": {"type": "string"},
                },
                "required": ["between", "description"],
                "additionalProperties": False,
            },
        },
        "recommended_action": {"type": "string"},
        "human_brief": {
            "type": "string",
            "description": "One paragraph for the human reviewer if this escalates.",
        },
    },
    ["summary", "conflicts", "recommended_action", "human_brief"],
)

RECONCILER_PROMPT = """You are the reconciliation layer of a compliance system in a regulated bank.

Three specialists have independently reviewed a document. Merge their findings into one
assessment.

Your priorities, in order:
1. Surface conflicts. If two specialists contradict each other, that is your most important
   output — never average it away or quietly prefer one.
2. Report calibrated confidence in the MERGED picture. Conflicting or low-confidence inputs
   must produce a low confidence here — do not inherit a specialist's certainty.
3. Recommend an action and write a brief for the human who may review this.

You do not set the stakes — firm policy already fixed those from the document type. You are
accountable to a human reviewer, not to a clean answer. An honest 'these two findings do not
line up and I cannot tell which is right' is a correct result."""


async def reconcile(
    case_id: str,
    document: str,
    routing: dict[str, Any],
    findings: dict[str, dict[str, Any]],
    stakes: Stakes,
) -> dict[str, Any]:
    prompt = f"""Reconcile these independent specialist reports.

<document>
{document}
</document>

<routing intent="{routing["intent"]}" document_type="{routing["document_type"]}" />

<specialist_reports>
{json.dumps(findings, indent=2)}
</specialist_reports>

Each report includes the specialist's own confidence. Weigh them accordingly."""

    return await run_agent(
        case_id=case_id,
        name="reconciliation_agent",
        model=ORCHESTRATOR_MODEL,
        system_prompt=RECONCILER_PROMPT,
        prompt=prompt,
        schema=RECONCILER_SCHEMA,
        input_summary=f"merge {len(findings)} specialist reports",
        stakes=stakes,
    )

## Step 4: The Human-in-the-Loop Checkpoint

Here is where most "human-in-the-loop" implementations quietly fail. They put the instruction in the prompt — *"ask before submitting"* — and call it a control. That is not a control; it is a request. The model can forget it, reason around it, or be talked out of it by a persuasive document.

A real checkpoint has to sit **outside** the model, where prompting cannot reach it.

The SDK's `can_use_tool` callback is exactly that seam. It fires *before* any tool executes, and whatever it returns is final:

- `PermissionResultAllow` — the tool runs.
- `PermissionResultDeny` — the tool does not run. The model is told why, and it cannot override it.

So we hang the failure budget off that callback. FC-A/B calls through. FC-C/D stops dead and waits for a human — and the block is enforced by Python, not by the model's good intentions.

> **One SDK detail that will bite you:** `can_use_tool` requires **streaming mode**. Pass a plain string as your prompt and the SDK raises `can_use_tool callback requires streaming mode`. The callback is part of a bidirectional control protocol — the CLI has to be able to call back *into* your process mid-run to ask permission, which a one-shot string prompt cannot support. The `user_message` helper below wraps a string into the async iterable the SDK expects.

In [12]:
async def user_message(text: str) -> AsyncIterator[dict[str, Any]]:
    """Wrap a prompt as a streaming message.

    Required whenever `can_use_tool` is set: the permission callback needs a
    bidirectional channel, which a plain string prompt does not provide.
    """
    yield {
        "type": "user",
        "message": {"role": "user", "content": text},
        "parent_tool_use_id": None,
        "session_id": "default",
    }

In [13]:
@dataclass
class ApprovalRequest:
    case_id: str
    tool_name: str
    tool_input: dict[str, Any]
    fc_tier: FCTier
    brief: str
    decision: str | None = None
    reviewer: str | None = None


@dataclass
class ApprovalQueue:
    """A stand-in for the review console a human compliance officer would actually use."""

    pending: list[ApprovalRequest] = field(default_factory=list)
    history: list[ApprovalRequest] = field(default_factory=list)

    def submit(self, request: ApprovalRequest) -> ApprovalRequest:
        self.pending.append(request)
        return request

    def decide(self, request: ApprovalRequest, decision: str, reviewer: str) -> ApprovalRequest:
        if decision not in {"approved", "rejected"}:
            raise ValueError("decision must be 'approved' or 'rejected'")
        request.decision = decision
        request.reviewer = reviewer
        self.pending.remove(request)
        self.history.append(request)
        return request


queue = ApprovalQueue()

### The regulated tool

`submit_to_core_banking` is the one action in this notebook with real-world consequences — it is the point of no return. We define it as an in-process MCP tool with the `@tool` decorator, so the SDK exposes it to the agent exactly as it would a production banking API.

The `executed` list is our tripwire: it records every submission that actually went through. At the end of the demos we assert against it to prove the FC-D block was real and not merely reported.

In [14]:
executed: list[dict[str, Any]] = []


@tool(
    "submit_to_core_banking",
    "Submit a final, irreversible decision to the core banking system.",
    {"case_id": str, "decision": str, "rationale": str},
)
async def submit_to_core_banking(args: dict[str, Any]) -> dict[str, Any]:
    """The point of no return: this is the call a compliance gate exists to guard."""
    executed.append(args)
    return {
        "content": [
            {
                "type": "text",
                "text": f"SUBMITTED case {args['case_id']}: {args['decision']}",
            }
        ]
    }


banking_server = create_sdk_mcp_server(
    name="core_banking",
    version="1.0.0",
    tools=[submit_to_core_banking],
)

SUBMIT_TOOL = "mcp__core_banking__submit_to_core_banking"

### The gate

`make_gate` builds the `can_use_tool` callback for one case, closing over that case's final tier. Read the branches in order — they *are* the failure budget, now with teeth:

- **FC-A / FC-B** → allow, and log it (FC-B's "automated with logging" is this line).
- **FC-C** → suspend the agent, push to the approval queue, and let a human decide. Approve and it proceeds; reject and it does not.
- **FC-D** → deny outright. No human is asked, because FC-D means the case should never have reached a tool call in the first place.

Every branch writes to the audit trail, including the human's name and verdict.

> ### ⚠️ The footgun that silently disables your gate
>
> **Do not put the guarded tool in `allowed_tools`.**
>
> `allowed_tools` is an **auto-approve** list, not an availability list. A tool named there is pre-approved, so the permission system never asks — and `can_use_tool` is **never called**. Your gate still exists, your FC-D branch still looks correct in review, and every single call sails straight through.
>
> This fails in the worst possible direction: silently, and only in production. Nothing errors. The audit trail even looks plausible. Meanwhile the trade you thought you blocked has settled.
>
> The tool is available because the MCP server provides it. Leave it out of `allowed_tools` so every call is routed through the callback, and use `disallowed_tools` to close off the built-in tools the agent might otherwise reach for as a workaround.
>
> Step 7 asserts against the `executed` list precisely so this class of bug cannot hide behind a reassuring log line.

We also pass `interrupt=False` on the denials. The tool is blocked either way — the difference is that the agent is *told* it was denied and reports it, rather than having its turn killed. A clean, reported denial is what you want in the audit trail.

In [15]:
def make_gate(case_id: str, tier: FCTier, brief: str, reviewer: "HumanReviewer"):
    """Build a can_use_tool callback that enforces the failure budget for one case."""

    async def gate(
        tool_name: str,
        tool_input: dict[str, Any],
        context: ToolPermissionContext,
    ) -> PermissionResultAllow | PermissionResultDeny:
        # Anything that isn't the regulated action is out of scope for this gate.
        if tool_name != SUBMIT_TOOL:
            return PermissionResultAllow()

        # FC-D: blocked outright. No human is asked — this must not reach an action at all.
        if tier is FCTier.D:
            audit.record(
                case_id=case_id,
                agent="fc_gate",
                input_summary=f"{tool_name} blocked at {tier.label}",
                output={"blocked": True, "tool_input": tool_input},
                fc_tier=tier,
                human_decision="escalated (not executed)",
            )
            print(f"  [GATE] {tier.label} — BLOCKED. Escalated to compliance; tool not executed.")
            return PermissionResultDeny(
                message=(
                    "BLOCKED by the FC-D failure budget: this action is high-stakes or "
                    "confidence is below 0.70. It has been escalated to a compliance officer. "
                    "Do not retry or attempt an alternative route — stop and report the block."
                ),
            )

        # FC-C: suspend and wait for a human.
        if tier.requires_human:
            request = queue.submit(
                ApprovalRequest(
                    case_id=case_id,
                    tool_name=tool_name,
                    tool_input=tool_input,
                    fc_tier=tier,
                    brief=brief,
                )
            )
            print(f"  [GATE] {tier.label} — paused, awaiting human approval...")
            decided = await reviewer.review(request)

            audit.record(
                case_id=case_id,
                agent="fc_gate",
                input_summary=f"{tool_name} held for review at {tier.label}",
                output={"tool_input": tool_input, "reviewer": decided.reviewer},
                fc_tier=tier,
                human_decision=decided.decision,
            )

            if decided.decision == "approved":
                print(f"  [GATE] approved by {decided.reviewer} — proceeding.")
                return PermissionResultAllow()

            print(f"  [GATE] rejected by {decided.reviewer} — tool not executed.")
            return PermissionResultDeny(
                message=f"Rejected by {decided.reviewer}. Do not retry.",
            )

        # FC-A / FC-B: automated. FC-B is logged for post-hoc review.
        audit.record(
            case_id=case_id,
            agent="fc_gate",
            input_summary=f"{tool_name} auto-approved at {tier.label}",
            output={"tool_input": tool_input},
            fc_tier=tier,
        )
        print(f"  [GATE] {tier.label} — auto-approved.")
        return PermissionResultAllow()

    return gate

### The reviewer

`HumanReviewer` stands in for the person a real deployment would page. It prints the brief exactly as a review console would render it, then applies a scripted verdict so the notebook runs unattended.

Swapping this class for a Slack round-trip or a ticket queue is the only change needed to make this production-shaped — the gate above does not care where the answer comes from, only that it arrives.

In [16]:
@dataclass
class HumanReviewer:
    """Simulates the compliance officer. Replace with a real console, Slack, or ticket queue."""

    name: str
    verdict: str = "approved"

    async def review(self, request: ApprovalRequest) -> ApprovalRequest:
        print("\n  ┌─ APPROVAL REQUIRED " + "─" * 58)
        print(f"  │ case      : {request.case_id}")
        print(f"  │ tier      : {request.fc_tier.label}")
        print(f"  │ action    : {request.tool_name}")
        print(f"  │ decision  : {request.tool_input.get('decision', '—')}")
        for i, line in enumerate(textwrap.wrap(request.brief, 64) or ["—"]):
            label = "brief     :" if i == 0 else " " * 11
            print(f"  │ {label} {line}")
        print("  └" + "─" * 78)

        await asyncio.sleep(0)  # a real reviewer round-trip awaits here
        decided = queue.decide(request, self.verdict, self.name)
        print(f"  → {self.name} {decided.decision} this request.\n")
        return decided

## Step 5: Assembling the Pipeline

Now we compose the layers. `process_document` is the whole system in one function, and the ordering encodes the governance:

1. **Route** → classify the document, then look up its stakes floor from `POLICY_FLOOR`. Provisional tier = stricter of the router's confidence tier and that floor.
2. **Run the specialists** — independent calls, at the case's stakes floor.
3. **Reconcile** → final tier = `max(provisional, reconciled)`, so the tier can only ratchet upward.
4. **Act** through the gate — the only path to the regulated tool, whatever the tier.

Every case runs the full pipeline, including one the router has already marked FC-D. That is deliberate here: the gate is the single enforcement point we want to study, and Demo 2 only demonstrates it if an FC-D case actually reaches a tool call.

In production you would short-circuit an FC-D routing decision immediately and skip the specialist calls — cheap escalation is a feature, and the FC-D path should be the least expensive one in the system. The trade-off is where enforcement lives: short-circuiting moves the block into the pipeline, whereas leaving it at the gate keeps every route to the tool passing through one auditable check.

The execution agent at the end is the only agent in the notebook with a tool. Note that it is told to submit — no prompt-level "check first", no "be careful". The gate does not need the model's cooperation, and that is the entire point.

In [17]:
EXECUTOR_PROMPT = f"""You are the execution agent for a regulated bank.

Submit the recommended decision to the core banking system using the
`{SUBMIT_TOOL}` tool, then report what happened in one sentence.

If the tool is denied, stop immediately and report the denial verbatim. Never attempt an
alternative route, a retry, or a workaround — a denial is a compliance control, not an
obstacle to solve."""


async def process_document(case_id: str, document: str, reviewer: HumanReviewer) -> dict[str, Any]:
    """Run one document through the full compliance pipeline."""
    print(f"══ {case_id} " + "═" * (76 - len(case_id)))

    # 1. Route, then look up the policy stakes floor from the document class.
    routing = await route(case_id, document)
    stakes = POLICY_FLOOR[routing["document_type"]]
    provisional = assign_tier(routing["confidence"], stakes)
    print(
        f"  router     : {routing['document_type']}  policy_stakes={stakes.name}  "
        f"conf={routing['confidence']:.2f}  -> {provisional.label}"
    )

    # 2. Run the specialists — independent calls, at the case's stakes floor.
    findings = await run_specialists(case_id, document, stakes)
    for name, finding in findings.items():
        print(f"  {name:<19}: conf={finding['confidence']:.2f}")

    # 3. Reconcile. The tier ratchets up, never down.
    merged = await reconcile(case_id, document, routing, findings, stakes)
    reconciled = assign_tier(merged["confidence"], stakes)
    final = max(provisional, reconciled)
    print(f"  reconciler : conf={merged['confidence']:.2f}  -> {reconciled.label}")
    print(f"  FINAL TIER : {final.label}  (conflicts: {len(merged['conflicts'])})")
    for conflict in merged["conflicts"]:
        print(f"    conflict : {' vs '.join(conflict['between'])} — {conflict['description']}")

    # 4. Act — every route to the tool passes through the gate.
    options = ClaudeAgentOptions(
        model=ORCHESTRATOR_MODEL,
        system_prompt=EXECUTOR_PROMPT,
        mcp_servers={"core_banking": banking_server},
        # NOTE: SUBMIT_TOOL is deliberately NOT in allowed_tools. Listing it there would
        # auto-approve it and can_use_tool would never fire — silently disabling the gate.
        # The MCP server is what makes the tool available; allowed_tools only pre-approves.
        disallowed_tools=["Bash", "Read", "Write", "Edit", "Task", "WebSearch", "WebFetch"],
        can_use_tool=make_gate(case_id, final, merged["human_brief"], reviewer),
        max_turns=6,
    )
    execution_prompt = (
        f"Case {case_id}. Recommended action: {merged['recommended_action']}\n"
        f"Rationale: {merged['summary']}\n\nSubmit this decision."
    )

    # can_use_tool requires streaming mode, so the prompt goes in as an async iterable.
    # Guard against a wedged subprocess with a timeout — but do NOT retry the executor:
    # a retry could re-submit an already-executed decision. The gate is the safety net here.
    async def _execute() -> str:
        summary = ""
        async for message in query(prompt=user_message(execution_prompt), options=options):
            if isinstance(message, ResultMessage):
                summary = (message.result or "").strip()
        return summary

    agent_summary = await asyncio.wait_for(_execute(), timeout=180.0)
    print(f"  agent      : {textwrap.shorten(agent_summary, 100)}\n")

    return {"case_id": case_id, "final_tier": final, "merged": merged}

## Step 6: Demo 1 — A Loan Application That Escalates (FC-C)

The document below is deliberately *borderline rather than damning* — the realistic middle of the distribution, where most real escalations live. A self-employed applicant with lumpy, partially-evidenced income, a deposit that is part savings and part family gift, and an affordability field left blank. Nothing here is fraud. It is exactly the kind of file a human underwriter should lay eyes on before the firm commits.

That is the point of FC-C: a **loan application is MEDIUM stakes by policy** — an underwriting recommendation a human ratifies — so the floor is FC-C regardless of how the numbers land. Watch the specialists weigh in, the reconciler produce a merged view, and the gate suspend the agent to wait for a human's sign-off.

In [18]:
LOAN_APPLICATION = """
MORTGAGE APPLICATION — REF MTG-2024-88213
Applicant:            J. Whitfield, age 34, self-employed contractor (2 years trading)
Stated gross income:  GBP 72,000 / year (variable; invoices attached for 9 of 24 months)
Requested loan:       GBP 300,000 over 25 years
Property value:       GBP 400,000
Deposit:              GBP 100,000 — GBP 60,000 savings, GBP 40,000 "family gift"
Existing credit:      GBP 9,200 across 2 facilities, all up to date
Affordability check:  [FIELD BLANK]
Adviser note:         "Income is lumpy but trending up. Worth a proper look."
"""

reviewer = HumanReviewer(name="A. Okafor (Compliance)", verdict="approved")
loan_result = await process_document("MTG-2024-88213", LOAN_APPLICATION, reviewer)

══ MTG-2024-88213 ══════════════════════════════════════════════════════════════


  router     : loan_application  policy_stakes=MEDIUM  conf=0.93  -> FC-C


  document_extractor : conf=0.95
  compliance_check   : conf=0.92
  risk_assessor      : conf=0.72


  reconciler : conf=0.85  -> FC-C
  FINAL TIER : FC-C  (conflicts: 2)
    conflict : compliance_check vs risk_assessor — Severity framing of the £40,000 undocumented 'family gift' deposit differs: compliance_check rates this a 'medium' severity breach, while risk_assessor's narrative treats it as a top-tier red flag comparable to the missing affordability check. Direction of concern is the same, but the weighting is inconsistent and should be reconciled by a human before deciding how hard to push on gift-letter documentation.
    conflict : compliance_check vs risk_assessor — Numerical inconsistency in estimated monthly mortgage repayment: compliance_check gives ~£1,440/month on an interest-only basis and explicitly states capital+interest would be higher; risk_assessor gives ~£1,450/month (£17,400/year) apparently as the full repayment-mortgage figure. These two numbers are nearly identical despite representing what should be different repayment structures — likely an unverified/incor

  [GATE] FC-C — paused, awaiting human approval...

  ┌─ APPROVAL REQUIRED ──────────────────────────────────────────────────────────
  │ case      : MTG-2024-88213
  │ tier      : FC-C
  │ action    : mcp__core_banking__submit_to_core_banking
  │ decision  : HOLD — Return to underwriting/adviser (no approval or decline). Require before any decision: (1) completed, documented affordability assessment per MCOB 11.6.1R including interest-rate stress test and income-drop scenario given self-employed/variable-income profile; (2) full self-employment income evidence (ideally 2-3 years of accounts/SA302s/tax returns) to replace the current 9-of-24-months invoices; (3) formal verification of the £40,000 family gift deposit portion — signed gift letter, confirmation of non-repayable status, and basic source-of-funds/AML check; (4) a structured, documented suitability rationale to replace the informal adviser note. Escalate to a human underwriter/compliance officer rather than resolving via fur

  agent      : The HOLD decision (returning MTG-2024-88213 to underwriting/adviser with the four specified [...]



### The audit trail for this case

This is the artefact the whole notebook exists to produce. Every agent that touched the case, what it saw, what it concluded, how confident it was, the tier that produced, and the human who signed off — reconstructable months later, which is precisely what MiFID II Art. 16(7) and BCBS 239 ask for.

In [19]:
audit.render("MTG-2024-88213")

[ 2] 2026-07-16T09:51:11+00:00  intent_router          conf=0.93  FC-B
     in : MORTGAGE APPLICATION — REF MTG-2024-88213 Applicant: J. Whitfield, age 34, self- [...]
     out: {"document_type": "loan_application", "intent": "The document requests [...]
[ 3] 2026-07-16T09:51:23+00:00  document_extractor     conf=0.95  FC-C
     in : Extract every stated fact, and list any field you would expect but cannot find.
     out: {"facts": [{"field": "Application Reference", "value": "MTG-2024-88213"}, [...]
[ 4] 2026-07-16T09:51:44+00:00  compliance_check       conf=0.92  FC-C
     in : Identify regulatory breaches or concerns, citing the specific regime.
     out: {"verdict": "fail", "breaches": [{"regulation": "FCA Consumer Duty (COBS 2.1R, 2A) [...]
[ 5] 2026-07-16T09:52:06+00:00  risk_assessor          conf=0.72  FC-C
     in : Assess the risk of acting on this document and list any red flags.
     out: {"risk_level": "high", "confidence": 0.72, "exposure": "**Primary exposure: [...]
[ 6]

## Step 7: Demo 2 — A Trade Instruction That Gets Blocked (FC-D)

Now the other end of the failure budget. This document has every hallmark of an authorised-push-payment fraud: a large round number, an urgency lever, a new unverified counterparty, and an explicit request to bypass the four-eyes control.

The interesting part is not that it looks suspicious. It is that **the tier does not depend on the agent noticing**. A `trade_instruction` is REGULATED by policy, so the stakes floor pins the case to FC-D no matter how confident any agent is — even a fully-persuaded agent that decided this trade was legitimate would still be blocked. The control does not route through the model's judgement, which is exactly what you want for the class of action that must never be automated.

Our reviewer is instructed to approve everything. FC-D does not ask them.

In [20]:
TRADE_INSTRUCTION = """
TRADE INSTRUCTION — REF TRD-2024-00417
Client:          Meridian Holdings SARL (onboarded 6 days ago)
Instruction:     SELL 100,000 shares VODAFONE GROUP PLC at market
Settlement:      Proceeds to Account 8837-2291, Bank of Valletta, third-party beneficiary
Value (approx):  GBP 12,000,000
Received via:    Email, 23:47 local time
Authorisation:   Single signatory — "second signatory unavailable, please proceed anyway"
Client note:     "Must execute before market open. Do not delay for the usual checks."
"""

# This reviewer approves everything they are shown — FC-D never shows them anything.
permissive_reviewer = HumanReviewer(name="B. Lindqvist (Ops)", verdict="approved")
trade_result = await process_document("TRD-2024-00417", TRADE_INSTRUCTION, permissive_reviewer)

══ TRD-2024-00417 ══════════════════════════════════════════════════════════════


  router     : trade_instruction  policy_stakes=REGULATED  conf=0.97  -> FC-D


  document_extractor : conf=0.72
  compliance_check   : conf=0.92
  risk_assessor      : conf=0.92


  reconciler : conf=0.88  -> FC-D
  FINAL TIER : FC-D  (conflicts: 0)


  [GATE] FC-D — BLOCKED. Escalated to compliance; tool not executed.


  agent      : The tool call was denied: **"BLOCKED by the FC-D failure budget: this action is high-stakes or [...]



### Proving the block was real

A printed "BLOCKED" is a claim, not evidence — and in a compliance system, the difference between those two words is the entire job.

This matters more than it looks. The most likely way this notebook could be wrong is not a crash; it is a gate that *reports* a block while the tool quietly runs anyway (put `SUBMIT_TOOL` in `allowed_tools` and you get exactly that — see the warning in Step 4). Every log line would still say BLOCKED.

So we do not trust the log. `executed` records what actually reached the core banking system, and we assert against it: the loan (FC-C, human-approved) should be there, and the trade (FC-D) must not be. If the gate ever regresses, this cell raises.

In [21]:
submitted_cases = [entry["case_id"] for entry in executed]

print("Submitted to core banking :", submitted_cases or "(nothing)")
print("Loan  final tier          :", loan_result["final_tier"].label)
print("Trade final tier          :", trade_result["final_tier"].label)
print()

assert "TRD-2024-00417" not in submitted_cases, "FC-D trade reached the banking system!"
print("FC-D trade never reached the core banking system — the gate held.")

if "MTG-2024-88213" in submitted_cases:
    print("FC-C loan was submitted only after a human approved it.")
else:
    print("FC-C loan was not submitted (the reviewer rejected it, or the agent stopped short).")

print(f"\nHuman decisions recorded: {[(r.case_id, r.decision) for r in queue.history]}")

Submitted to core banking : ['MTG-2024-88213']
Loan  final tier          : FC-C
Trade final tier          : FC-D

FC-D trade never reached the core banking system — the gate held.
FC-C loan was submitted only after a human approved it.

Human decisions recorded: [('MTG-2024-88213', 'approved')]


### The complete audit trail

Both cases, every agent, in order — the full record from a single append-only table.

In [22]:
audit.render()

[ 1] 2026-07-16T09:51:00+00:00  self-test              conf=1.00  FC-A
     in : verify the append-only guarantee
     out: {"note": "this row must be immutable"}
[ 2] 2026-07-16T09:51:11+00:00  intent_router          conf=0.93  FC-B
     in : MORTGAGE APPLICATION — REF MTG-2024-88213 Applicant: J. Whitfield, age 34, self- [...]
     out: {"document_type": "loan_application", "intent": "The document requests [...]
[ 3] 2026-07-16T09:51:23+00:00  document_extractor     conf=0.95  FC-C
     in : Extract every stated fact, and list any field you would expect but cannot find.
     out: {"facts": [{"field": "Application Reference", "value": "MTG-2024-88213"}, [...]
[ 4] 2026-07-16T09:51:44+00:00  compliance_check       conf=0.92  FC-C
     in : Identify regulatory breaches or concerns, citing the specific regime.
     out: {"verdict": "fail", "breaches": [{"regulation": "FCA Consumer Duty (COBS 2.1R, 2A) [...]
[ 5] 2026-07-16T09:52:06+00:00  risk_assessor          conf=0.72  FC-C
     in : 

## Conclusion

We built a compliance-aware agent that can answer the model risk officer's question from the introduction: *why did it decide this, who signed off, and how sure was it?*

Four ideas did the work, and each one moved a control **out of the prompt and into the system**:

| Idea | The failure it prevents |
| --- | --- |
| **Failure budget (FC-A/B/C/D)** as a typed primitive | Risk staying an unenforceable comment in a system prompt |
| **Independent specialists** with isolated context | A single confident opinion that nothing can contradict |
| **Reconciliation** that ratchets tiers upward only | Confidence downstream quietly overriding caution upstream |
| **`can_use_tool` as the checkpoint** | "Ask before submitting" being a request the model can reason around |
| **Append-only audit trail** enforced by triggers | Evidence that can be rewritten after the fact |

The one that generalises furthest is the third and fourth together. A human-in-the-loop control that lives in the prompt is not a control — the model can forget it, or be argued out of it by a sufficiently urgent-sounding document. Putting the gate in `can_use_tool` means the block is enforced by Python, and no amount of persuasion in the input reaches it. That is the difference between an agent that *usually* asks permission and one you can put in front of a regulator.

Notice too that FC-D never consulted a human. That is deliberate: a tier that means *"this should not have got here"* should not be answerable by whoever happens to be on call at 23:47 — which, as Demo 2 showed, is exactly when these requests arrive.

### Where to take it next

- **Calibrate the thresholds.** The 0.95/0.85/0.70 cut-offs are a starting point, not a law. Backtest them against decisions you already know the outcome of, and tune until the FC-C queue is a volume your reviewers can actually absorb.
- **Make the reviewer real.** `HumanReviewer` is the one deliberately fake component. Swap it for Slack, PagerDuty, or your case management system — the gate is indifferent to where the verdict comes from.
- **Persist the audit trail.** Move `AuditLog` to a durable, access-controlled store with retention that matches your regime (MiFID II wants five years). Keep the triggers.
- **Short-circuit FC-D at the router.** We deliberately ran the full pipeline so that Demo 2 reached the gate. In production, escalate as soon as the router says FC-D and skip the specialist calls — just keep the gate in place as the backstop for everything that does proceed.
- **Add drift monitoring.** Track the FC-tier distribution over time. A sudden fall in FC-C escalations rarely means the model improved — it usually means confidence drifted upward while accuracy did not, and SR 11-7 expects you to catch that.

### Related notebooks

- [01 — The chief of staff agent](https://github.com/anthropics/claude-cookbooks/blob/main/claude_agent_sdk/01_The_chief_of_staff_agent.ipynb): subagents, hooks, and orchestration primitives.
- [03 — The site reliability agent](https://github.com/anthropics/claude-cookbooks/blob/main/claude_agent_sdk/03_The_site_reliability_agent.ipynb): read-write tools and hook-based guardrails for autonomous remediation.
- [06 — The vulnerability detection agent](https://github.com/anthropics/claude-cookbooks/blob/main/claude_agent_sdk/06_The_vulnerability_detection_agent.ipynb): another high-stakes domain where confidence signalling matters.